# 02 DIP figures (S2 and the DIP pipeline)

Figures for the report, Gate 1 of the R&D plan. Logic lives in `src/geoinfer/viz/dip_figures.py`;
this notebook only loads data, calls it, and saves PNGs to `docs/figures/s2_*.png`.
Run headless with `make figures-nb` (part of `make figures`).

Inputs: the 200-tile working subset (`~/geo-data/subset`), the DVC-tracked statistics
(`data/processed/stats/`), the co-registration table and D4.

In [ ]:
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import yaml
from rasterio.windows import Window

from geoinfer.io.raster import RasterReader
from geoinfer.preprocess.sar import sar_chain
from geoinfer.preprocess.stats import BandHistogram
from geoinfer.viz import dip_figures as dfig

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
DATA = yaml.safe_load((ROOT / "configs/data.yaml").read_text())
PRE = yaml.safe_load((ROOT / "configs/preprocess.yaml").read_text())
TILING = yaml.safe_load((ROOT / "configs/tiling.yaml").read_text())
EVAL = yaml.safe_load((ROOT / "configs/eval.yaml").read_text())
STATS_DIR = ROOT / PRE["stats"]["out_dir"]
STATS = json.loads((STATS_DIR / "stats.json").read_text())
STRIP = json.loads((STATS_DIR / PRE["sar"]["strip_stats"]["out"]).read_text())
SUBSET = Path(DATA["subset"]["root"]).expanduser()
OUT = ROOT / "docs/figures"
RX = re.compile(DATA["tile_id_regex"])
ENL = STATS["sar"]["enl"]["value"]
METRICS = {}


def tiles(modality):
    return {m.group(1): f for f in (SUBSET / modality).glob("*.tif") if (m := RX.search(f.name))}


def read(path):
    with RasterReader(path) as r:
        return r.read(Window(0, 0, r.info.width, r.info.height)), r.info


SAR, OPT = tiles(DATA["modalities"]["sar"]), tiles(DATA["modalities"]["optical"])
print(len(SAR), "subset tiles; ENL", ENL)

## 1. Optical normalisation: per-tile min-max vs global stretch (S2)

In [ ]:
# two edge-to-edge tiles of one strip (tiles of a strip share one grid)
by_strip = defaultdict(list)
for t in sorted(OPT):
    by_strip[t.split("_tile_")[0]].append(t)
pair = None
for ids in by_strip.values():
    infos = {t: read(OPT[t])[1].transform for t in ids}
    for i, a in enumerate(ids):
        for b in ids[i + 1 :]:
            dc = round((infos[b].c - infos[a].c) / infos[a].a)
            dr = round((infos[b].f - infos[a].f) / infos[a].e)
            if (abs(dc), abs(dr)) in {(900, 0), (0, 900)} and dc >= 0 and dr >= 0:
                pair = (a, b, (dr, dc))
                break
        if pair:
            break
    if pair:
        break
a, b, off = pair
(ra, _), (rb, _) = read(OPT[a]), read(OPT[b])
fig, m = dfig.optical_seam_figure(ra.data, rb.data, off, STATS["optical"])
fig.savefig(OUT / "s2_optical_seam.png")
METRICS["optical_seam"] = m | {"tiles": [a, b]}
m

## 2. Histograms before / after normalisation (train split)

In [ ]:
opt_h = BandHistogram.load(STATS_DIR / "hist_optical.npz")
sar_h = {
    "db": BandHistogram.load(STATS_DIR / "hist_sar_db.npz"),
    "db_lee5": BandHistogram.load(STATS_DIR / "hist_sar_lee5.npz"),
    "db_lee7": BandHistogram.load(STATS_DIR / "hist_sar_lee7.npz"),
}
fig, m = dfig.histogram_figure(opt_h, STATS["optical"], sar_h, STATS["sar"]["arms"])
fig.savefig(OUT / "s2_histograms.png")
METRICS["histograms"] = m
m

## 3. SAR speckle: linear vs dB vs Lee 5 vs Lee 7 (ENL, EPI)

In [ ]:
best = None
for t in sorted(SAR):
    rd, _ = read(SAR[t])
    valid = np.all(rd.data != 0, axis=0)
    if valid.mean() < 0.6:  # subset SAR tiles are 50-80 % valid
        continue
    lin = 10 ** (rd.data[0].astype(float) / 10)
    try:
        r, c = dfig.homogeneous_window(lin, valid, 32)
    except ValueError:
        continue
    blk = lin[r : r + 32, c : c + 32]
    cv = blk.std() / blk.mean()
    if best is None or cv < best[0]:
        best = (cv, t, r, c)
_, t, r, c = best
rd, _ = read(SAR[t])
valid = np.all(rd.data != 0, axis=0)
# the 300 px crop holding the window with the most valid pixels
cands = [
    (max(min(r + 16 - y, 600), 0), max(min(c + 16 - x, 600), 0))
    for y in range(32, 284, 24)
    for x in range(32, 284, 24)
]
r0, c0 = max(cands, key=lambda rc: valid[rc[0] : rc[0] + 300, rc[1] : rc[1] + 300].sum())
crop = (slice(None), slice(r0, r0 + 300), slice(c0, c0 + 300))
db = rd.data[crop]
fig, m = dfig.sar_patch_figure(db, np.all(db != 0, axis=0), ENL)
fig.savefig(OUT / "s2_sar_patch.png")
METRICS["sar_patch"] = m | {"tile": t, "crop": [r0, c0, 300]}
m

## 4. Strip calibration (decision D4)

In [ ]:
arm = STRIP["arms"]["db_lee7_strip"]
strips = {k: v for k, v in arm["strips"].items() if v is not None}
means = {k: v["mean"] for k, v in strips.items()}
glob = STATS["sar"]["arms"]["db_lee7"]
sub_strips = sorted({t.split("_tile_")[0] for t in SAR} & set(strips), key=lambda s: means[s][0])
picked = [sub_strips[0], sub_strips[len(sub_strips) // 2], sub_strips[-1]]
samples = {}
for s in picked:
    t = next(t for t in sorted(SAR) if t.startswith(s))
    rd, _ = read(SAR[t])
    valid = np.all(rd.data != 0, axis=0)
    samples[s] = sar_chain(rd.data, valid, 7, ENL)[0][valid]
fig, m = dfig.strip_calibration_figure(means, glob["mean"], samples, strips, glob)
fig.savefig(OUT / "s2_strip_calibration.png")
METRICS["strip_calibration"] = m | {"fallback_strips": len(arm["strips"]) - len(strips)}
m

## 5. Co-registration: SAR and optical edge maps

In [ ]:
shifts = pd.read_csv(OUT / "coreg_shifts.csv")
t = shifts.sort_values("response", ascending=False).tile.iloc[0]
(rs, _), (ro, _) = read(SAR[t]), read(OPT[t])
valid = np.all(rs.data != 0, axis=0) & np.any(ro.data != 0, axis=0)
cc = PRE["coreg"]
fig, m = dfig.coreg_overlay_figure(rs.data, ro.data, valid, cc["sobel_ksize"], cc["blur_sigma"])
fig.savefig(OUT / "s2_coreg_overlay.png")
shift = shifts.set_index("tile").loc[t, ["dx_px", "dy_px"]].tolist()
METRICS["coreg_overlay"] = m | {"tile": t, "shift_px": shift}
m

## 6. Tiling grid overlay on D4

In [ ]:
d4 = Path(DATA["scenes"]["d4"])
down, rows = 4, 1536
with rasterio.open(d4 / "d4_optical.tif") as s:
    w = s.width
    ov = s.read([1, 2, 3], window=Window(0, 0, w, rows), out_shape=(3, rows // down, w // down))
v = ov.astype(float)
lo, hi = np.percentile(v[v > 0], [2, 98])
img = np.clip((np.moveaxis(v, 0, -1) - lo) / (hi - lo), 0, 1)
fig, m = dfig.tiling_overlay_figure(
    img, 1 / down, (rows, w), TILING["tile"], TILING["overlap"], EVAL["border_band_px"]
)
fig.savefig(OUT / "s2_tiling_overlay.png")
METRICS["tiling_overlay"] = m
m

## Numbers shown in the figures

In [ ]:
(OUT / "s2_metrics.json").write_text(json.dumps(METRICS, indent=2, default=str) + "\n")
print(json.dumps(METRICS, indent=2, default=str))